In [1]:
import requests #Pour faire des requêtes HTTP
from bs4 import BeautifulSoup #Pour extraire et manipuler des données HTML
import pandas as pd #Manipulation de données
import time

In [2]:
categories_generiques = {"Accueil", "Femmes", "Hommes", "Enfants", "Vêtements"}

variables=["url", "etat", "matiere", "couleur", "date", "categorie", "pays", "likes", "prix", "prix_total"]

df_vinted = pd.DataFrame(columns=variables)
df_vinted.head()

,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total


In [3]:
url="https://www.vinted.fr/catalog?catalog[]=1904&brand_ids[]=12&page=1&time=1778184660"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')


Requête réussie; Code: 200


In [4]:
urls = [link.get('href') for link in soup.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 
print (urls[0])
print(len(urls))

https://www.vinted.fr/items/8878361301-sans-manche-zara?referrer=catalog
96


In [49]:
url=urls[0]
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [50]:
#scrapper l'état'
get_etat=soup.select_one('[itemprop="status"]')
etat = get_etat.get_text(" ", strip=True) if get_etat else None

etat

'Très bon état'

In [64]:
#scrapper la matière
get_matiere=soup.select_one('[itemprop="material"]')
if get_matiere:
    matiere = get_matiere.get_text(" ", strip=True)

matiere

In [52]:
#scrapper la couleur
get_couleur=soup.select_one('[itemprop="color"]')
couleur = get_couleur.get_text(" ", strip=True) if get_couleur else None

couleur

'Bleu clair'

In [53]:
#scrapper date
get_date=soup.select_one('[itemprop="upload_date"]')
date = get_date.get_text(" ", strip=True) if get_date else None

date

'Il y a 3 heures'

In [54]:
#scrapper le prix
get_prix = soup.select_one('[data-testid="item-price"]')
get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

prix = get_prix.get_text(" ", strip=True) if get_prix else None
prix_total=get_prix_total.get_text(" ", strip=True) if get_prix else None

print(prix, prix_total)

8,00 € 9,10 €


In [55]:
#scrapper catégories

categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
]

categorie = None

for cat in categories:
    if cat not in categories_generiques:
        categorie = cat
        break

print(categorie)

Jeans


In [56]:
#scrapper pays
get_pays = soup.select_one('[data-testid="seller-location"]')
pays=get_pays.get_text(" ", strip=True) if get_prix else None

pays


'Paris, France'

In [57]:
#scrapper likes
get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
likes=get_likes.get_text(" ", strip=True) if get_likes else None

likes

'20'

In [5]:
def scrapping(link):
    response = requests.get(link,headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, 'html.parser')
    infos=["status", "material", "color", "upload_date" ]
    #info
    values=[link]
    for info in infos:
        get_info=soup.select_one(f'[itemprop="{info}"]')
        if get_info:
            info = get_info.get_text(" ", strip=True)
        else:
            info=None
        values.append(info)
    #prix
    get_prix = soup.select_one('[data-testid="item-price"]')
    get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

    prix = get_prix.get_text(" ", strip=True) if get_prix else None
    prix_total=get_prix_total.get_text(" ", strip=True) if get_prix_total else None
    #categories
    categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
    ]

    categorie = None

    for cat in categories:
        if cat not in categories_generiques:
            categorie = cat
            break
    #pays
    get_pays = soup.select_one('[data-testid="seller-location"]')
    if get_pays:
        pays=get_pays.get_text(" ", strip=True)
    else:
        pays=None

    #likes
    get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
    if get_likes:
        likes=get_likes.get_text(" ", strip=True)
    else:
        likes=None
            
    values+=[categorie, pays, likes, prix, prix_total]

    return values
    

In [73]:
scrapping(url)

['Très bon état',
 None,
 'Bleu clair',
 'Il y a 3 heures',
 '8,00\xa0€',
 '9,10\xa0€',
 'Jeans',
 'Paris, France',
 '27']

In [11]:
#construction du dataframe
df_vinted_femme=df_vinted.copy()
n=0
for i in urls:
    time.sleep(3)
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_femme.loc[len(df_vinted_femme)] = ligne
        n+=1
        print(f"  [{n}/{len(urls)}]")
    
df_vinted_femme.head()

  [1/96]
  [2/96]
  [3/96]
  [4/96]
  [5/96]
  [6/96]
  [7/96]
  [8/96]
  [9/96]
  [10/96]
  [11/96]
  [12/96]
  [13/96]
  [14/96]
  [15/96]
  [16/96]
  [17/96]
  [18/96]
  [19/96]
  [20/96]
  [21/96]
  [22/96]
  [23/96]
  [24/96]
  [25/96]
  [26/96]
  [27/96]
  [28/96]
  [29/96]
  [30/96]
  [31/96]
  [32/96]
  [33/96]
  [34/96]
  [35/96]
  [36/96]
  [37/96]
  [38/96]
  [39/96]
  [40/96]
  [41/96]
  [42/96]
  [43/96]
  [44/96]
  [45/96]
  [46/96]
  [47/96]
  [48/96]
  [49/96]
  [50/96]
  [51/96]
  [52/96]
  [53/96]
  [54/96]
  [55/96]
  [56/96]
  [57/96]
  [58/96]
  [59/96]
  [60/96]
  [61/96]
  [62/96]
  [63/96]
  [64/96]
  [65/96]
  [66/96]
  [67/96]
  [68/96]
  [69/96]
  [70/96]
  [71/96]
  [72/96]
  [73/96]
  [74/96]
  [75/96]
  [76/96]
  [77/96]
  [78/96]
  [79/96]
  [80/96]
  [81/96]
  [82/96]
  [83/96]
  [84/96]
  [85/96]
  [86/96]
  [87/96]
  [88/96]
  [89/96]
  [90/96]
  [91/96]
  [92/96]
  [93/96]
  [94/96]
  [95/96]
  [96/96]


,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total
0,https://www.vinted.fr/items/8878361301-sans-ma...,Neuf sans étiquette,None,"Crème, Noir",Il y a 5 heures,Sweats et sweats à capuche,"Combourg, France",21,"7,00 €","7,70 €"
1,https://www.vinted.fr/items/8878321166-robe-lo...,Très bon état,Coton,Blanc,Il y a 5 heures,Robes,"Fresnes, France",58,"22,00 €","23,80 €"
2,https://www.vinted.fr/items/8866538965-jean-bl...,Très bon état,None,Bleu,Il y a un jour,Jeans,None,24,"15,00 €","16,40 €"
3,https://www.vinted.fr/items/8875283926-total-l...,Neuf sans étiquette,"Coton, Lin",Beige,Il y a 8 heures,Blazers et tailleurs,None,62,"35,00 €","37,45 €"
4,https://www.vinted.fr/items/8878420953-robe-lo...,Très bon état,None,Gris,Il y a 5 heures,Robes,None,38,"22,00 €","23,97 €"


In [12]:
df_vinted_femme['collection']="femme"
#df_vinted_femme.to_csv("vinted_data_femme.csv", index=False, encoding="utf-8-sig")



In [13]:
url_homme="https://www.vinted.fr/catalog?catalog[]=5&brand_ids[]=12&page=1&time=1778222504"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_homme = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [14]:
urls_homme = [link.get('href') for link in soup_homme.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 
print(len(urls_homme))

96


In [15]:
df_vinted_homme=df_vinted.copy()
n=0
for i in urls_homme:
    time.sleep(2)
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_homme.loc[len(df_vinted_homme)] = ligne
        n+=1
        print(f"  [{n}/{len(urls)}]")
    
df_vinted_homme.head()

  [1/96]
  [2/96]
  [3/96]
  [4/96]
  [5/96]
  [6/96]
  [7/96]
  [8/96]
  [9/96]
  [10/96]
  [11/96]
  [12/96]
  [13/96]
  [14/96]
  [15/96]
  [16/96]
  [17/96]
  [18/96]
  [19/96]
  [20/96]
  [21/96]
  [22/96]
  [23/96]
  [24/96]
  [25/96]
  [26/96]
  [27/96]
  [28/96]
  [29/96]
  [30/96]
  [31/96]
  [32/96]
  [33/96]
  [34/96]
  [35/96]
  [36/96]
  [37/96]
  [38/96]
  [39/96]
  [40/96]
  [41/96]
  [42/96]
  [43/96]
  [44/96]
  [45/96]
  [46/96]
  [47/96]
  [48/96]
  [49/96]
  [50/96]
  [51/96]
  [52/96]
  [53/96]
  [54/96]
  [55/96]
  [56/96]
  [57/96]
  [58/96]
  [59/96]
  [60/96]
  [61/96]
  [62/96]
  [63/96]
  [64/96]
  [65/96]
  [66/96]
  [67/96]
  [68/96]
  [69/96]
  [70/96]
  [71/96]
  [72/96]
  [73/96]
  [74/96]
  [75/96]
  [76/96]
  [77/96]
  [78/96]
  [79/96]
  [80/96]
  [81/96]
  [82/96]
  [83/96]
  [84/96]
  [85/96]
  [86/96]
  [87/96]
  [88/96]
  [89/96]
  [90/96]
  [91/96]
  [92/96]
  [93/96]
  [94/96]
  [95/96]
  [96/96]


,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total
0,https://www.vinted.fr/items/8878361301-sans-ma...,Neuf sans étiquette,None,"Crème, Noir",Il y a 5 heures,Sweats et sweats à capuche,"Combourg, France",23,"7,00 €","7,84 €"
1,https://www.vinted.fr/items/8880509953-bomber-...,Très bon état,None,Kaki,Il y a 16 minutes,Manteaux et vestes,"Lisbon, Portugal",None,"10,00 €","10,95 €"
2,https://www.vinted.fr/items/8878321166-robe-lo...,Très bon état,Coton,Blanc,Il y a 5 heures,Robes,"Fresnes, France",59,"22,00 €","23,97 €"
3,https://www.vinted.fr/items/8871389949-pantalo...,Neuf sans étiquette,None,"Rose, Blanc",Il y a 13 heures,Pantalons et leggings,"Barcelona, Espagne",8,"14,00 €","15,40 €"
4,https://www.vinted.fr/items/8880433843-jean-pa...,Très bon état,None,Marine,Il y a une heure,Jeans,None,15,"5,00 €","5,53 €"


In [16]:
df_vinted_homme['collection']="homme"
#df_vinted_homme.to_csv("vinted_data_homme.csv", index=False, encoding="utf-8-sig")

In [17]:
url_fille="https://www.vinted.fr/catalog?brand_ids[]=12&page=1&time=1778223434&catalog[]=1195"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_fille = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [18]:
urls_fille = [link.get('href') for link in soup_fille.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 

In [19]:
df_vinted_fille=df_vinted.copy()
n=0
for i in urls_fille:
    time.sleep(2)
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_fille.loc[len(df_vinted_fille)] = ligne
        n+=1
        print(f"  [{n}/{len(urls)}]")

  [1/96]
  [2/96]
  [3/96]
  [4/96]
  [5/96]
  [6/96]
  [7/96]
  [8/96]
  [9/96]
  [10/96]
  [11/96]
  [12/96]
  [13/96]
  [14/96]
  [15/96]
  [16/96]
  [17/96]
  [18/96]
  [19/96]
  [20/96]
  [21/96]
  [22/96]
  [23/96]
  [24/96]
  [25/96]
  [26/96]
  [27/96]
  [28/96]
  [29/96]
  [30/96]
  [31/96]
  [32/96]
  [33/96]
  [34/96]
  [35/96]
  [36/96]
  [37/96]
  [38/96]
  [39/96]
  [40/96]
  [41/96]
  [42/96]
  [43/96]
  [44/96]
  [45/96]
  [46/96]
  [47/96]
  [48/96]
  [49/96]
  [50/96]
  [51/96]
  [52/96]
  [53/96]
  [54/96]
  [55/96]
  [56/96]
  [57/96]
  [58/96]
  [59/96]
  [60/96]
  [61/96]
  [62/96]
  [63/96]
  [64/96]
  [65/96]
  [66/96]
  [67/96]
  [68/96]
  [69/96]
  [70/96]
  [71/96]
  [72/96]
  [73/96]
  [74/96]
  [75/96]
  [76/96]
  [77/96]
  [78/96]
  [79/96]
  [80/96]
  [81/96]
  [82/96]
  [83/96]
  [84/96]
  [85/96]
  [86/96]
  [87/96]
  [88/96]
  [89/96]
  [90/96]
  [91/96]
  [92/96]
  [93/96]
  [94/96]
  [95/96]
  [96/96]


In [20]:
df_vinted_fille['collection']="fille"
#df_vinted_fille.to_csv("vinted_data_fille.csv", index=False, encoding="utf-8-sig")

In [21]:
url_garcon ="https://www.vinted.fr/catalog?brand_ids[]=12&page=1&time=1778224020&catalog[]=1194"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_garcon = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [22]:
urls_garcon = [link.get('href') for link in soup_garcon.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 

In [24]:
df_vinted_garcon=df_vinted.copy()
n=0
for i in urls_garcon:
    time.sleep(1)
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_garcon.loc[len(df_vinted_garcon)] = ligne
        n+=1
        print(f"  [{n}/{len(urls)}]")

  [1/96]
  [2/96]
  [3/96]
  [4/96]
  [5/96]
  [6/96]
  [7/96]
  [8/96]
  [9/96]
  [10/96]
  [11/96]
  [12/96]
  [13/96]
  [14/96]
  [15/96]
  [16/96]
  [17/96]
  [18/96]
  [19/96]
  [20/96]
  [21/96]
  [22/96]
  [23/96]
  [24/96]
  [25/96]
  [26/96]
  [27/96]
  [28/96]
  [29/96]
  [30/96]
  [31/96]
  [32/96]
  [33/96]
  [34/96]
  [35/96]
  [36/96]
  [37/96]
  [38/96]
  [39/96]
  [40/96]
  [41/96]
  [42/96]
  [43/96]
  [44/96]
  [45/96]
  [46/96]
  [47/96]
  [48/96]
  [49/96]
  [50/96]
  [51/96]
  [52/96]
  [53/96]
  [54/96]
  [55/96]
  [56/96]
  [57/96]
  [58/96]
  [59/96]
  [60/96]
  [61/96]
  [62/96]
  [63/96]
  [64/96]
  [65/96]
  [66/96]
  [67/96]
  [68/96]
  [69/96]
  [70/96]
  [71/96]
  [72/96]
  [73/96]
  [74/96]
  [75/96]
  [76/96]
  [77/96]
  [78/96]
  [79/96]
  [80/96]
  [81/96]
  [82/96]
  [83/96]
  [84/96]
  [85/96]
  [86/96]
  [87/96]
  [88/96]
  [89/96]
  [90/96]
  [91/96]
  [92/96]
  [93/96]
  [94/96]
  [95/96]
  [96/96]


In [25]:
df_vinted_garcon['collection']="garcon"
#df_vinted_garcon.to_csv("vinted_data_garcon.csv", index=False, encoding="utf-8-sig")

In [26]:
df_total = pd.concat([df_vinted_femme, df_vinted_homme, df_vinted_fille, df_vinted_garcon], ignore_index=True)
df_total.to_csv("vinted_data.csv", index=False, encoding="utf-8-sig")